# Fix HTML issues
Автоматическая проверка и исправление сырых ampersand, дублированных meta и распространённые HTML-ошибки.

In [ ]:
import os, re, json

root = r'd:\bi\Documents\Мои веб-узлы\b-i'
report_path = os.path.join(root, 'html_report.txt')
if not os.path.exists(report_path):
    raise FileNotFoundError(report_path)
raw_amp = re.compile(r'&(?![A-Za-z]+;|#\d+;|#x[0-9A-Fa-f]+;)')
meta_x = re.compile(r'(<meta\b[^>]*http-equiv=["]X-UA-Compatible["][^>]*>)', re.I)
meta_v = re.compile(r'(<meta\b[^>]*name=["]viewport["][^>]*>)', re.I)

with open(report_path, 'r', encoding='utf-8', errors='replace') as f:
    lines = [line.strip() for line in f if line.startswith('FILE:')]
paths = [line[5:].strip() for line in lines]
log_lines = []
for rel in paths:
    path = os.path.join(root, os.path.relpath(rel, root))
    if not os.path.exists(path):
        log_lines.append(f'MISSING: {rel}')
        continue
    text = open(path, 'r', encoding='utf-8', errors='replace').read()
    orig = text
    changed = False
    details = []
    amp_matches = raw_amp.findall(text)
    if amp_matches:
        text = raw_amp.sub('&amp;', text)
        changed = True
        details.append(f'raw_amp={len(amp_matches)}')
    x_matches = list(meta_x.finditer(text))
    if len(x_matches) > 1:
        keep = x_matches[0].span()
        remove_spans = [m.span() for m in x_matches[1:]]
        new_text = []
        last = 0
        for start, end in remove_spans:
            new_text.append(text[last:start])
            last = end
        new_text.append(text[last:])
        text = ''.join(new_text)
        changed = True
        details.append(f'X-UA-Compatible removed={len(x_matches)-1}')
    v_matches = list(meta_v.finditer(text))
    if len(v_matches) > 1:
        remove_spans = [m.span() for m in v_matches[1:]]
        new_text = []
        last = 0
        for start, end in remove_spans:
            new_text.append(text[last:start])
            last = end
        new_text.append(text[last:])
        text = ''.join(new_text)
        changed = True
        details.append(f'viewport removed={len(v_matches)-1}')
    if changed and text != orig:
        with open(path, 'w', encoding='utf-8', newline='') as out:
            out.write(text)
        log_lines.append(f'FIXED: {rel} -> {
.join(details)}')
    else:
        log_lines.append(f'NO CHANGE: {rel}')

log_path = os.path.join(root, 'fix_html_issues_notebook.log')
with open(log_path, 'w', encoding='utf-8') as logf:
    logf.write('
'.join(log_lines))

# Verification
verify_path = os.path.join(root, 'fix_html_after_report.txt')
with open(verify_path, 'w', encoding='utf-8') as vf:
    vf.write('verification results
')
    for rel in paths:
        path = os.path.join(root, os.path.relpath(rel, root))
        if not os.path.exists(path):
            vf.write(f'MISSING: {rel}
')
            continue
        text = open(path, 'r', encoding='utf-8', errors='replace').read()
        ra = len(raw_amp.findall(text))
        xa = len(meta_x.findall(text))
        va = len(meta_v.findall(text))
        vf.write(f'{rel} raw_amp={ra} X-UA-Compatible={xa} viewport={va}
')

print('Done. Log:', log_path, 'Verify:', verify_path)